In [1]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
%matplotlib inline

In [2]:
words = open("names.txt", "r").read().splitlines()

In [3]:
len(words)

32033

In [4]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [5]:
chars = sorted(list(set(''.join(words)))) 
chars

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [6]:
# Build the mappings
stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0
itos = {ch: i for i, ch in stoi.items()}

itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [7]:
# Build the dataset
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]


In [8]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [9]:
X.shape, Y.shape, X.dtype, Y.dtype

(torch.Size([228146, 3]), torch.Size([228146]), torch.int64, torch.int64)

In [10]:
gen = torch.Generator().manual_seed(123)

In [11]:
C = torch.randn((27, 2), generator=gen)

In [12]:
C[5]

tensor([ 0.6984, -1.4097])

In [13]:
C[X].shape

torch.Size([228146, 3, 2])

In [14]:
X[1]

tensor([0, 0, 5])

In [15]:
C[X[1]]

tensor([[ 0.3374, -0.1778],
        [ 0.3374, -0.1778],
        [ 0.6984, -1.4097]])

In [16]:
# Embedding
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [17]:
# Define the parameters
W1 = torch.randn(6, 100, generator=gen)
b1 = torch.randn(100, generator=gen)
W2 = torch.randn(100, 27, generator=gen)
b2 = torch.randn(27, generator=gen)
parameters = [C, W1, b1, W2, b2]


In [18]:
for p in parameters:
    p.requires_grad = True

In [19]:
# Training loop
epochs = 100

for epoch in range(epochs):
    # construct mini batches
    batch_size = torch.randint(0, X.shape[0], (32,))

    # forward pass
    emb = C[X[batch_size]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[batch_size])
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item()}")

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    for p in parameters:
        p.data += -0.1 * p.grad

Epoch 0: Loss = 14.3963623046875
Epoch 10: Loss = 10.709432601928711
Epoch 20: Loss = 7.508938789367676
Epoch 30: Loss = 6.242143154144287
Epoch 40: Loss = 5.072078227996826
Epoch 50: Loss = 4.954698085784912
Epoch 60: Loss = 3.8500542640686035
Epoch 70: Loss = 4.516032695770264
Epoch 80: Loss = 2.9144296646118164
Epoch 90: Loss = 3.0363667011260986


In [20]:
# Overall loss calculation
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)

loss.item()

3.510833501815796

In [21]:
# Function to build the training, validation, and test sets
def build_dataset(words, block_size):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(f"X shape: {X.shape}, Y shape: {Y.shape}")
    return X, Y

In [22]:
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

In [23]:
print(f"Total words: {len(words)}, Training: {n1}, Validation: {n2-n1}, Test: {len(words)-n2}")

Total words: 32033, Training: 25626, Validation: 3203, Test: 3204


In [24]:
# Split the dataset into training, validation, and test sets
X_train, Y_train = build_dataset(words[:n1], 3)
X_val, Y_val = build_dataset(words[n1:n2], 3)
X_test, Y_test = build_dataset(words[n2:], 3)

X shape: torch.Size([182625, 3]), Y shape: torch.Size([182625])
X shape: torch.Size([22655, 3]), Y shape: torch.Size([22655])
X shape: torch.Size([22866, 3]), Y shape: torch.Size([22866])


In [25]:
# Define the parameters
W1 = torch.randn(6, 100, generator=gen)
b1 = torch.randn(100, generator=gen)
W2 = torch.randn(100, 27, generator=gen)
b2 = torch.randn(27, generator=gen)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [26]:
# Training loop
epochs = 1000

for epoch in range(epochs):
    # construct mini batches
    batch_size = torch.randint(0, X_train.shape[0], (32,))

    # forward pass
    emb = C[X_train[batch_size]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_train[batch_size])
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item()}")

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    for p in parameters:
        p.data += -0.1 * p.grad

Epoch 0: Loss = 17.312849044799805
Epoch 100: Loss = 3.0056817531585693
Epoch 200: Loss = 3.8056023120880127
Epoch 300: Loss = 3.305816411972046
Epoch 400: Loss = 2.8463234901428223
Epoch 500: Loss = 2.748401165008545
Epoch 600: Loss = 3.0665066242218018
Epoch 700: Loss = 2.8089847564697266
Epoch 800: Loss = 2.579472780227661
Epoch 900: Loss = 2.5166256427764893


In [27]:
# Evaluate on the validation set
emb = C[X_val]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_val)

loss.item()

2.6641550064086914

In [47]:
# Hyperparameter tuning - using a larger embedding size 
# Define the parameters
C = torch.randn((27, 6), generator=gen)  # Increased embedding size
W1 = torch.randn(18, 150, generator=gen)
b1 = torch.randn(150, generator=gen)
W2 = torch.randn(150, 27, generator=gen)
b2 = torch.randn(27, generator=gen)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [48]:
# Training loop
epochs = 5000

for epoch in range(epochs + 1):
    # construct mini batches
    batch_size = torch.randint(0, X_train.shape[0], (32,))

    # forward pass
    emb = C[X_train[batch_size]]
    h = torch.tanh(emb.view(-1, 18) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_train[batch_size])
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item()}")

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    for p in parameters:
        p.data += -0.1 * p.grad

Epoch 0: Loss = 25.07272720336914
Epoch 100: Loss = 7.8300676345825195
Epoch 200: Loss = 5.057823181152344
Epoch 300: Loss = 3.3736910820007324
Epoch 400: Loss = 3.546619176864624
Epoch 500: Loss = 3.8599660396575928
Epoch 600: Loss = 3.663806200027466
Epoch 700: Loss = 3.7536394596099854
Epoch 800: Loss = 3.7075209617614746
Epoch 900: Loss = 3.2127373218536377
Epoch 1000: Loss = 2.667865037918091
Epoch 1100: Loss = 2.6123669147491455
Epoch 1200: Loss = 2.6895594596862793
Epoch 1300: Loss = 2.53379487991333
Epoch 1400: Loss = 2.542741298675537
Epoch 1500: Loss = 2.541538953781128
Epoch 1600: Loss = 2.5648107528686523
Epoch 1700: Loss = 2.79776668548584
Epoch 1800: Loss = 2.9932315349578857
Epoch 1900: Loss = 2.9575374126434326
Epoch 2000: Loss = 2.756014108657837
Epoch 2100: Loss = 3.143061876296997
Epoch 2200: Loss = 2.7999017238616943
Epoch 2300: Loss = 2.688647508621216
Epoch 2400: Loss = 2.9070026874542236
Epoch 2500: Loss = 3.1159439086914062
Epoch 2600: Loss = 2.4752273559570312


In [49]:
# Evaluate on the validation set
emb = C[X_val]
h = torch.tanh(emb.view(-1, 18) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y_val)

loss.item()

2.6023075580596924

In [50]:
# sample from the model

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=gen).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

acortiazeri.
krolen.
fryd.
jarero.
quclerghphorsrellian.
jiy.
shmensaighton.
jaygoluwa.
linhod.
nuleoger.
lolyan.
iha.
nnami.
makororsewerdileyri.
zoi.
zhrere.
suma.
suna.
toudrreleyoho.
jassor.
